# A1.11 · Cascading hallucination

**Function A — Securing AI Architectures → The Agentic Reference Architecture, and Every Risk It Carries**  ·  *Security of AI*

Builds on **[A1.10 · Rogue agents in a multi-agent system](https://spbreed.github.io/cyber-commons/lessons/A1.10.html)**.

| | |
|---|---|
| Open-source tooling | Inspect |
| Open-weight models | GLM-4.6 |
| Frontier models | Claude Haiku 4.5 |

> **Runs anywhere.** Every line of code is in this notebook — nothing to install, nothing to clone, no API key, no network. Standard library only, so it works on a Kaggle kernel with the internet switched off — and where a lesson involves a model, the same code calls an open-weight endpoint or a frontier API when you configure one.

## 1 · The hook

Agent one is 90% accurate, which sounds fine. Agent two consumes its output as fact, and agent three consumes that. By the third hop the confident wrong answer has been repeated enough times that it reads like corroboration.

## 2 · The framework

```
   agent 1        agent 2        agent 3        report
   90% right ---> takes as ----> takes as ----> "three sources agree"
                  fact           fact

   error compounds: 0.9 -> 0.81 -> 0.73
   confidence compounds the other way, because repetition reads as corroboration
```

**OWASP T5 — Cascading Hallucination Attacks. LLM09 — Misinformation.**

The **model** component produces confident text. Sometimes the text is wrong.
That is a known property, and on its own it is a quality problem rather than a
security one.

It becomes a security problem when the architecture has more than one step,
because an unverified claim from step one is an *input* to step two. And inputs
are not re-examined — that is the point of a pipeline.

Watch what happens to a single fabrication as it travels:

**Hop one.** "I could not find a CVE for this dependency, it is probably fine."
Hedged, and the hedge is visible.

**Hop two.** The next agent summarises: "dependency has no known CVEs."
The hedge is gone. Nothing lied — summarising removes qualifiers, that is what
summarising is.

**Hop three.** "Dependency verified clean." Now it is a finding, with the
confidence of something that was checked, and no field anywhere records that
nobody checked anything.

The security consequence is that **confidence rises as evidence disappears**,
which is exactly backwards. And it is not limited to accidents: an attacker who
can inject one plausible claim early gets it laundered into an established fact
by your own pipeline, which is why this sits in the threat taxonomy rather than
in a quality backlog.

> **Where this lands on the reference architecture.**
>
> ```
> ingress -> orchestrator -> agent_runtime -> model
>                                |              |
>                          messaging        tools / mcp
>                                |              |
>                       knowledge / memory   egress
>            identity + policy wrap every call · observability records it
> ```

## 3 · The risk, realised

One hedged guess, three hops, and the confidence it acquires on the way.

In [ ]:
def summarise(claim, confidence):
    """Each hop compresses. Compression removes qualifiers first - they are the
    least information-dense part of a sentence."""
    for hedge in ("I could not find", "probably", "appears to", "it seems"):
        if hedge in claim:
            claim = " ".join(claim.replace(hedge, "").split())
            confidence = min(1.0, confidence + 0.3)     # certainty is what survives
    return claim.strip(", "), round(confidence, 2)

ORIGINAL = "I could not find a CVE for libfoo, it is probably fine"
claim, conf = ORIGINAL, 0.2
provenance = ["model guess, unverified"]

print(f"{'hop':>4}  {'confidence':>11}  claim")
print(f"{0:>4}  {conf:>11.2f}  {claim}")
for hop in (1, 2, 3):
    claim, conf = summarise(claim, conf)
    if hop >= 2:
        provenance = []                       # the source field is not carried on
    print(f"{hop:>4}  {conf:>11.2f}  {claim}")

print(f"\nprovenance recorded at hop 3: {provenance or 'none'}")
print(f"confidence at hop 0: 0.20   at hop 3: {conf}")
print()
print("Nothing lied. Every hop did its job. The claim gained certainty at the")
print("exact rate it lost evidence, and by hop three it reads like a finding")
print("someone verified.")
print()
print("An attacker who lands one plausible claim early gets it laundered into")
print("an established fact by your own pipeline - for free.")
assert conf >= 0.8 and not provenance

## What you just proved

A hedged guess at confidence 0.2 becomes a confident claim above 0.8 in three hops, while the provenance field empties — confidence rising at exactly the rate evidence disappears.

## Your turn

Take a finding your pipeline produced and try to walk it back to the step that first asserted it. If you cannot reach a step that checked something, you have found a cascade rather than a finding.

---

**Next → [A1.12 · Resource overload](https://spbreed.github.io/cyber-commons/lessons/A1.12.html)**

[All lessons](https://spbreed.github.io/cyber-commons/lessons/) · [This lesson's page](https://spbreed.github.io/cyber-commons/lessons/A1.11.html) · [Source](https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/labs/notebooks/A1.11.ipynb)

*Cyber Commons — a free, open commons for Cyber AI.*